# AMR-ALERT: UKHSA/JANIS forecasting workflow

This notebook implements the AMR-ALERT forecasting workflow used for England (UKHSA) and Japan (JANIS) surveillance analyses.

**Data sources**
- England: UK Health Security Agency (UKHSA) Data Dashboard API.
- Japan: `janis_compact_blood_named_validated.csv`, a processed aggregate dataset derived from publicly available JANIS surveillance reports.

Place `janis_compact_blood_named_validated.csv` in the same directory as this notebook before running Japan analyses.


In [ ]:
#@title AMR Forecast automation — Cell-1
# Project setup and imports
import os
import pandas as pd
import numpy as np

from scipy.stats import linregress
from sklearn.metrics import r2_score

import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("AMR Forecast automation setup complete.")

In [ ]:
#@title SetUp Parameters Cell-2
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# ------------------------------
# England / UKHSA verified choices
# ------------------------------
UKHSA_CHOICES = {
    "Escherichia coli": [
        "Amikacin",
        "Carbapenems",
        "Ciprofloxacin",
        "Co-amoxiclav",
        "Gentamicin",
        "Piperacillin with tazobactam",
        "Third Generation Cephalosporins",
    ],
    "Klebsiella pneumoniae": [
        "Gentamicin",
        "Carbapenems",
        "Piperacillin with tazobactam",
        "Ciprofloxacin",
        "Amikacin",
        "Co-amoxiclav",
        "Third Generation Cephalosporins",
    ],
    "Pseudomonas aeruginosa": [
        "Amikacin",
        "Ciprofloxacin",
        "Carbapenems",
        "Gentamicin",
        "Ceftazidime",
        "Piperacillin with tazobactam",
    ],
    "Enterococcus faecium": [
        "Glycopeptides",
    ],
    "Enterococcus faecalis": [
        "Glycopeptides",
    ],
    "Streptococcus pneumoniae": [
        "Penicillin",
        "Macrolides",
    ],
        "Escherichia coli bacteriuria": [
        "nitrofurantoin",
        "trimethoprim"
    ],
}

# ------------------------------
# Japan / JANIS choices from compact validated CSV
# ------------------------------
JANIS_COMPACT_PATH = "janis_compact_blood_named_validated.csv"

if not os.path.exists(JANIS_COMPACT_PATH):
    raise FileNotFoundError(
        "JANIS compact CSV not found. "
        "Please place janis_compact_blood_named_validated.csv in the current working directory first."
    )

janis_inventory_df = pd.read_csv(JANIS_COMPACT_PATH)

required_cols = {"country", "pathogen", "antibiotic"}
missing_cols = required_cols - set(janis_inventory_df.columns)

if missing_cols:
    raise ValueError(f"JANIS compact CSV is missing required columns: {missing_cols}")

JANIS_CHOICES = (
    janis_inventory_df
    .query("country == 'Japan'")
    .groupby("pathogen")["antibiotic"]
    .apply(lambda x: sorted(x.dropna().unique().tolist()))
    .to_dict()
)

# ------------------------------
# Combined country routing choices
# ------------------------------
COUNTRY_CHOICES = {
    "England": UKHSA_CHOICES,
    "Japan": JANIS_CHOICES,
}

country_dropdown = widgets.Dropdown(
    options=list(COUNTRY_CHOICES.keys()),
    value="England",
    description="Country:",
    style={"description_width": "initial"},
)

pathogen_dropdown = widgets.Dropdown(
    options=sorted(COUNTRY_CHOICES[country_dropdown.value].keys()),
    description="Pathogen:",
    style={"description_width": "initial"},
)

antibiotic_dropdown = widgets.Dropdown(
    options=COUNTRY_CHOICES[country_dropdown.value][pathogen_dropdown.value],
    description="Antibiotic:",
    style={"description_width": "initial"},
)

threshold_dropdown = widgets.Dropdown(
    options=[10, 20, 30, 40, 50],
    value=20,
    description="Threshold (%):",
    style={"description_width": "initial"},
)

def update_pathogen_options(change=None):
    selected_country = country_dropdown.value
    pathogen_options = sorted(COUNTRY_CHOICES[selected_country].keys())

    pathogen_dropdown.options = pathogen_options
    pathogen_dropdown.value = pathogen_options[0]

    update_antibiotic_options()

def update_antibiotic_options(change=None):
    selected_country = country_dropdown.value
    selected_pathogen = pathogen_dropdown.value

    antibiotic_options = COUNTRY_CHOICES[selected_country][selected_pathogen]
    antibiotic_dropdown.options = antibiotic_options
    antibiotic_dropdown.value = antibiotic_options[0]

country_dropdown.observe(update_pathogen_options, names="value")
pathogen_dropdown.observe(update_antibiotic_options, names="value")

# ------------------------------
# Button to lock selected values for Cell 3
# ------------------------------
set_inputs_button = widgets.Button(
    description="Set Inputs",
    button_style="success", # 'success', 'info', 'warning', 'danger' or ''
    tooltip="Click to set inputs for the next cell",
)

input_status = widgets.Output()

@set_inputs_button.on_click
def on_button_clicked(b):
    with input_status:
        clear_output()
        global pathogen, antibiotic, country, theta
        pathogen = pathogen_dropdown.value
        antibiotic = antibiotic_dropdown.value
        country = country_dropdown.value
        theta = threshold_dropdown.value / 100
        print(f"Inputs set: Country={country}, Pathogen={pathogen}, Antibiotic={antibiotic}, Threshold={theta*100}%")

# ------------------------------
# Remove left-side descriptions
# ------------------------------
for w in [
    country_dropdown,
    pathogen_dropdown,
    antibiotic_dropdown,
    threshold_dropdown,
]:
    w.description = ""

# ------------------------------
# Adjust widths
# ------------------------------
country_dropdown.layout = widgets.Layout(width="120px")
pathogen_dropdown.layout = widgets.Layout(width="200px")
antibiotic_dropdown.layout = widgets.Layout(width="200px")
threshold_dropdown.layout = widgets.Layout(width="100px")

set_inputs_button.layout = widgets.Layout(width="110px")

# ------------------------------
# Build labelled controls
# ------------------------------
country_box = widgets.VBox([
    widgets.HTML("<b>Country</b>"),
    country_dropdown
])

pathogen_box = widgets.VBox([
    widgets.HTML("<b>Pathogen</b>"),
    pathogen_dropdown
])

antibiotic_box = widgets.VBox([
    widgets.HTML("<b>Antibiotic</b>"),
    antibiotic_dropdown
])

threshold_box = widgets.VBox([
    widgets.HTML("<b>Threshold (%)</b>"),
    threshold_dropdown
])

# Blank header keeps the button aligned with the dropdown row
button_box = widgets.VBox([
    widgets.HTML("&nbsp;"),
    set_inputs_button
])

display(
    widgets.HBox(
        [
            country_box,
            pathogen_box,
            antibiotic_box,
            threshold_box,
            button_box,
        ],
        layout=widgets.Layout(
            align_items="flex-start",
            gap="15px"
        )
    )
)

display(input_status)


In [ ]:
#@title Forecasting Cell-3
import os
from openpyxl import load_workbook

import requests
import pandas as pd
import numpy as np
from scipy.stats import linregress, t as student_t
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt


# ============================================================
# 1. VERIFIED SOURCE REGISTRY
# ============================================================

ukhsa_urls = {
    "Escherichia coli": (
        "https://api.ukhsa-dashboard.data.gov.uk/"
        "themes/infectious_disease/"
        "sub_themes/antimicrobial_resistance/"
        "topics/E-coli/"
        "geography_types/Nation/"
        "geographies/England/"
        "metrics/"
        "e-coli_testing_bacteraemiaPercentResistantRollingMonth"
        "?format=json&page_size=365"
    ),

    "Klebsiella pneumoniae": (
        "https://api.ukhsa-dashboard.data.gov.uk/"
        "themes/infectious_disease/"
        "sub_themes/antimicrobial_resistance/"
        "topics/K-pneumoniae/"
        "geography_types/Nation/"
        "geographies/England/"
        "metrics/"
        "k-pneumoniae_testing_bacteraemiaPercentResistantRollingMonth"
        "?format=json&page_size=365"
    ),

    "Pseudomonas aeruginosa": (
        "https://api.ukhsa-dashboard.data.gov.uk/"
        "themes/infectious_disease/"
        "sub_themes/antimicrobial_resistance/"
        "topics/P-aeruginosa/"
        "geography_types/Nation/"
        "geographies/England/"
        "metrics/"
        "p-aeruginosa_testing_bacteraemiaPercentResistantRollingMonth"
        "?format=json&page_size=365"
    ),

    "Streptococcus pneumoniae": (
        "https://api.ukhsa-dashboard.data.gov.uk/"
        "themes/infectious_disease/"
        "sub_themes/antimicrobial_resistance/"
        "topics/S-pneumoniae/"
        "geography_types/Nation/"
        "geographies/England/"
        "metrics/"
        "s-pneumoniae_testing_bacteraemiaPercentResistantRollingMonth"
        "?format=json&page_size=365"
    ),

    "Enterococcus faecium": (
        "https://api.ukhsa-dashboard.data.gov.uk/"
        "themes/infectious_disease/"
        "sub_themes/antimicrobial_resistance/"
        "topics/E-faecium/"
        "geography_types/Nation/"
        "geographies/England/"
        "metrics/"
        "e-faecium_testing_bacteraemiaPercentResistantRollingMonth"
        "?format=json&page_size=365"
    ),

    "Enterococcus faecalis": (
        "https://api.ukhsa-dashboard.data.gov.uk/"
        "themes/infectious_disease/"
        "sub_themes/antimicrobial_resistance/"
        "topics/E-faecalis/"
        "geography_types/Nation/"
        "geographies/England/"
        "metrics/"
        "e-faecalis_testing_bacteraemiaPercentResistantRollingMonth"
        "?format=json&page_size=365"
    ),

    "Escherichia coli bacteriuria": (
        "https://api.ukhsa-dashboard.data.gov.uk/"
        "themes/infectious_disease/"
        "sub_themes/antimicrobial_resistance/"
        "topics/E-coli/"
        "geography_types/Nation/"
        "geographies/England/"
        "metrics/"
        "e-coli_testing_bacteriuriaPercentResistantRollingMonthAllMedFac"
        "?format=json&page_size=365"
    )
}

antibiotic_maps = {
    "Escherichia coli": {
        "third-generation cephalosporins": "Third Generation Cephalosporins",
        "ciprofloxacin": "Ciprofloxacin",
        "co-amoxiclav": "Co-amoxiclav",
        "gentamicin": "Gentamicin",
        "amikacin": "Amikacin",
        "carbapenems": "Carbapenems",
        "piperacillin with tazobactam": "Piperacillin with tazobactam"
    },

    "Klebsiella pneumoniae": {
        "third-generation cephalosporins": "Third Generation Cephalosporins",
        "ciprofloxacin": "Ciprofloxacin",
        "co-amoxiclav": "Co-amoxiclav",
        "gentamicin": "Gentamicin",
        "amikacin": "Amikacin",
        "carbapenems": "Carbapenems",
        "piperacillin with tazobactam": "Piperacillin with tazobactam"
    },

    "Pseudomonas aeruginosa": {
        "ciprofloxacin": "Ciprofloxacin",
        "piperacillin with tazobactam": "Piperacillin with tazobactam",
        "amikacin": "Amikacin",
        "carbapenems": "Carbapenems",
        "ceftazidime": "Ceftazidime",
        "gentamicin": "Gentamicin"
    },

    "Streptococcus pneumoniae": {
        "penicillin": "Penicillin",
        "macrolides": "Macrolides"
    },

    "Enterococcus faecium": {
        "glycopeptides": "Glycopeptides"
    },

    "Enterococcus faecalis": {
        "glycopeptides": "Glycopeptides"
    },

    "Escherichia coli bacteriuria": {
        "nitrofurantoin": "Nitrofurantoin",
        "trimethoprim": "Trimethoprim"
    }
}


# ============================================================
# 2. SOURCE-SPECIFIC LOADER AND STANDARDIZER
# ============================================================

def load_ukhsa(url):
    all_results = []

    while url:
        response = requests.get(url)
        response.raise_for_status()

        json_data = response.json()

        all_results.extend(json_data["results"])
        url = json_data["next"]

    return pd.DataFrame(all_results)

def standardize_ukhsa(raw_data, selected_source):

    data = raw_data[
        (raw_data["stratum"] == selected_source["stratum"]) &
        (raw_data["in_reporting_delay_period"] == False)
    ].copy()

    data = data[
        ["year", "month", "date", "metric_value"]
    ].rename(
        columns={"metric_value": "resistance_percent"}
    )

    data["date"] = pd.to_datetime(data["date"])
    data["R"] = data["resistance_percent"] / 100

    data = (
        data
        .dropna()
        .sort_values("date")
        .reset_index(drop=True)
    )

    if len(data) == 0:
        raise ValueError(
            "The selected source returned no usable resistance data."
        )

    if ((data["R"] <= 0) | (data["R"] >= 1)).any():
        raise ValueError(
            "Resistance fractions must be strictly between 0 and 1 "
            "for the log-odds model."
        )

    data["source_name"] = selected_source["source_name"]
    data["pathogen"] = selected_source["pathogen"]
    data["antibiotic"] = selected_source["antibiotic"]
    data["country"] = selected_source["country"]

    return data

def load_janis(url=None):
    janis_compact_file = "janis_compact_blood_named_validated.csv"

    if not os.path.exists(janis_compact_file):
        raise FileNotFoundError(
            "Named validated JANIS compact dataset not found. "
            "Please place janis_compact_blood_named_validated.csv in the current working directory first."
        )

    return pd.read_csv(janis_compact_file)

def standardize_janis(raw_data, selected_source):

    data = raw_data[
        (raw_data["country"].str.lower() == selected_source["country"].lower()) &
        (raw_data["pathogen"].str.lower() == selected_source["pathogen"].lower()) &
        (raw_data["antibiotic"].str.lower() == selected_source["antibiotic"].lower()) &
        (raw_data["specimen"].str.lower() == selected_source["specimen"].lower())
    ].copy()

    data["date"] = pd.to_datetime(data["date"])

    data = (
        data
        .dropna(subset=["date", "R"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    if len(data) == 0:
        raise ValueError("JANIS compact dataset returned no usable data.")

    if ((data["R"] <= 0) | (data["R"] >= 1)).any():
        raise ValueError(
            "Resistance fractions must be strictly between 0 and 1 "
            "for the log-odds model."
        )

    return data

# ============================================================
# 3. BUILD SOURCE REGISTRY
# ============================================================

registry_rows = []

for pathogen_name, ukhsa_url in ukhsa_urls.items():
    for antibiotic_name, ukhsa_stratum in antibiotic_maps[pathogen_name].items():
        registry_rows.append({
            "source_name": "UKHSA",
            "pathogen": pathogen_name,
            "antibiotic": antibiotic_name,
            "country": "England",
            "url": ukhsa_url,
            "stratum": ukhsa_stratum,
            "loader": load_ukhsa,
            "standardizer": standardize_ukhsa,
            "status": "supported"
        })

janis_compact_file = "janis_compact_blood_named_validated.csv"

if os.path.exists(janis_compact_file):
    janis_inventory = pd.read_csv(janis_compact_file)

    janis_combinations = (
        janis_inventory[
            ["country", "pathogen", "antibiotic", "specimen", "pathogen_code", "antibiotic_code"]
        ]
        .drop_duplicates()
        .sort_values(["pathogen", "antibiotic"])
    )

    for _, row in janis_combinations.iterrows():
        registry_rows.append({
            "source_name": "JANIS",
            "pathogen": row["pathogen"],
            "antibiotic": row["antibiotic"],
            "country": row["country"],
            "url": None,
            "specimen": row["specimen"],
            "pathogen_code": row["pathogen_code"],
            "antibiotic_code": row["antibiotic_code"],
            "loader": load_janis,
            "standardizer": standardize_janis,
            "status": "supported"
        })
else:
    print(
        "Warning: JANIS compact file not found. "
        "Japan combinations will not be registered."
    )

source_registry = pd.DataFrame(registry_rows)


# ============================================================
# 4. VALIDATE, ROUTE, LOAD, AND STANDARDIZE
# ============================================================

match = source_registry[
    (source_registry["pathogen"].str.lower() == pathogen.lower()) &
    (source_registry["antibiotic"].str.lower() == antibiotic.lower()) &
    (source_registry["country"].str.lower() == country.lower()) &
    (source_registry["status"] == "supported")
]

if len(match) == 0:
    raise ValueError(
        "This pathogen-antibiotic-country combination "
        "is not currently supported."
    )

selected_source = match.iloc[0]

loader = selected_source["loader"]
standardizer = selected_source["standardizer"]

raw_data = loader(selected_source["url"])

data = standardizer(
    raw_data,
    selected_source
)


# ============================================================
# 4B. DESCRIPTIVE DATA COMPLETENESS SUMMARY
# ============================================================

date_min = data["date"].min()
date_max = data["date"].max()
year_span = (date_max - date_min).days / 365.25
n_observations = len(data)
distinct_years = data["date"].dt.year.nunique()

if selected_source["source_name"] == "UKHSA":
    reporting_delay_rows_removed = len(raw_data[
        (raw_data["stratum"] == selected_source["stratum"]) &
        (raw_data["in_reporting_delay_period"] == True)
    ])

    expected_periods = len(pd.date_range(start=date_min, end=date_max, freq="MS"))
    missing_periods = max(expected_periods - n_observations, 0)
    missing_fraction = (
        missing_periods / expected_periods if expected_periods > 0 else np.nan
    )
    reporting_frequency = "monthly"

elif selected_source["source_name"] == "JANIS":
    reporting_delay_rows_removed = "not applicable"
    expected_periods = int(date_max.year - date_min.year + 1)
    missing_periods = max(expected_periods - n_observations, 0)
    missing_fraction = (
        missing_periods / expected_periods if expected_periods > 0 else np.nan
    )
    reporting_frequency = "annual"

else:
    raise ValueError("Unknown source type.")

# ============================================================
# 5. MODEL FIT
# ============================================================

data["O"] = data["R"] / (1 - data["R"])
data["log_O"] = np.log(data["O"])

start_date = data["date"].min()
data["t_years"] = (data["date"] - start_date).dt.days / 365.25

t = data["t_years"]
y = data["log_O"]

fit = linregress(t, y)

g_R = fit.slope
log_O0 = fit.intercept

data["log_O_pred"] = log_O0 + g_R * t
data["O_pred"] = np.exp(data["log_O_pred"])
data["R_pred"] = data["O_pred"] / (1 + data["O_pred"])

r2 = r2_score(data["log_O"], data["log_O_pred"])


# ============================================================
# 6. MODEL-BASED AMRTF FORECAST GATE
# ============================================================

MIN_OBSERVATIONS = 8
MIN_DISTINCT_YEARS = 3
MIN_RESIDUAL_DF = 4
MIN_DISTINCT_R_VALUES = 3

BOOTSTRAP_PASS_FRACTION = 0.80
BOOTSTRAP_CAUTION_FRACTION = 0.60

n_parameters = 2
residual_df = n_observations - n_parameters
distinct_R_values = data["R"].nunique()

slope_se = fit.stderr

if (
    np.isfinite(slope_se)
    and residual_df > 0
):
    slope_ci_lower = g_R - 1.96 * slope_se
    slope_ci_upper = g_R + 1.96 * slope_se
else:
    slope_ci_lower = np.nan
    slope_ci_upper = np.nan

basic_estimability = (
    n_observations >= MIN_OBSERVATIONS
    and distinct_years >= MIN_DISTINCT_YEARS
    and residual_df >= MIN_RESIDUAL_DF
    and distinct_R_values >= MIN_DISTINCT_R_VALUES
    and data["t_years"].nunique() >= 3
    and np.isfinite(g_R)
    and np.isfinite(log_O0)
    and np.isfinite(slope_se)
)

positive_growth = (
    np.isfinite(g_R)
    and g_R > 0
)

positive_growth_supported = (
    positive_growth
    and np.isfinite(slope_ci_lower)
    and slope_ci_lower > 0
)

latest_date = data["date"].max()
latest_row = data.loc[
    data["date"] == latest_date
].iloc[0]

latest_R = latest_row["R"]
latest_t = latest_row["t_years"]

# Date on which the notebook is run.
# Europe/London is used so the date matches the user's local date.
run_date = (
    pd.Timestamp.now(tz="Europe/London")
    .normalize()
    .tz_localize(None)
)

target_log_odds = np.log(
    theta / (1 - theta)
)

threshold_already_crossed = (
    latest_R >= theta
)

T_theta_years = None
crossing_date = None
remaining_from_run_date = None
valid_future_crossing = False

if (
    basic_estimability
    and positive_growth
    and not threshold_already_crossed
):
    T_theta_years = (
        target_log_odds - log_O0
    ) / g_R

    if np.isfinite(T_theta_years):
        crossing_date = (
            start_date
            + pd.to_timedelta(
                T_theta_years * 365.25,
                unit="D"
            )
        )

        remaining_from_run_date = (
            crossing_date - run_date
        ).days / 365.25

        valid_future_crossing = (
            np.isfinite(remaining_from_run_date)
            and remaining_from_run_date >= 0
        )

# ============================================================
# 7. FORECAST GRID
# ============================================================

forecast_horizon_years = 5

plot_t = np.linspace(
    0,
    latest_t + forecast_horizon_years,
    300
)

plot_dates = [
    start_date + pd.to_timedelta(x * 365.25, unit="D")
    for x in plot_t
]

plot_log_O = log_O0 + g_R * plot_t
plot_O = np.exp(plot_log_O)
plot_R = plot_O / (1 + plot_O)


# ============================================================
# 8. RESIDUAL BOOTSTRAP AND FINAL GATE CLASSIFICATION
# ============================================================

n_bootstrap = 500
rng = np.random.default_rng(123)

residuals = (
    data["log_O"]
    - data["log_O_pred"]
)

bootstrap_R = []
bootstrap_crossing_dates = []
bootstrap_remaining_years = []
bootstrap_intercepts = [] # New: Store intercepts for bootstrap
bootstrap_slopes = []     # New: Store slopes for bootstrap

valid_bootstrap_fits = 0

bootstrap_assessed = (
    basic_estimability
    and positive_growth
    and not threshold_already_crossed
    and valid_future_crossing
)

if bootstrap_assessed:

    for _ in range(n_bootstrap):

        sampled_residuals = rng.choice(
            residuals,
            size=len(residuals),
            replace=True
        )

        boot_y = (
            data["log_O_pred"]
            + sampled_residuals
        )

        boot_fit = linregress(
            t,
            boot_y
        )

        boot_g_R = boot_fit.slope
        boot_log_O0 = boot_fit.intercept

        if not (
            np.isfinite(boot_g_R)
            and np.isfinite(boot_log_O0)
            and boot_g_R > 0
        ):
            # Append NaN or a placeholder if the fit was not valid
            bootstrap_intercepts.append(np.nan)
            bootstrap_slopes.append(np.nan)
            continue

        valid_bootstrap_fits += 1

        # Append valid fit parameters
        bootstrap_intercepts.append(boot_log_O0)
        bootstrap_slopes.append(boot_g_R)

        boot_plot_log_O = (
            boot_log_O0
            + boot_g_R * plot_t
        )

        boot_plot_O = np.exp(
            boot_plot_log_O
        )

        boot_plot_R = (
            boot_plot_O
            / (1 + boot_plot_O)
        )

        bootstrap_R.append(
            boot_plot_R
        )

        boot_T_theta = (
            target_log_odds
            - boot_log_O0
        ) / boot_g_R

        if not np.isfinite(
            boot_T_theta
        ):
            continue

        boot_crossing_date = (
            start_date
            + pd.to_timedelta(
                boot_T_theta * 365.25,
                unit="D"
            )
        )

        boot_remaining_from_run_date = (
            boot_crossing_date
            - run_date
        ).days / 365.25

        if (
            np.isfinite(
                boot_remaining_from_run_date
            )
            and boot_remaining_from_run_date >= 0
        ):
            bootstrap_crossing_dates.append(
                boot_crossing_date
            )

            bootstrap_remaining_years.append(
                boot_remaining_from_run_date
            )

bootstrap_R = np.asarray(
    bootstrap_R
)

if len(bootstrap_R) > 0:

    lower_band = np.percentile(
        bootstrap_R,
        2.5,
        axis=0
    )

    upper_band = np.percentile(
        bootstrap_R,
        97.5,
        axis=0
    )

else:
    lower_band = None
    upper_band = None

if bootstrap_assessed:

    valid_bootstrap_fraction = (
        len(bootstrap_remaining_years)
        / n_bootstrap
    )

else:
    valid_bootstrap_fraction = None

forecast_reasons = []

if not basic_estimability:

    forecast_status = "FAIL"

    forecast_reasons.append(
        "insufficient information for stable model estimation"
    )

elif threshold_already_crossed:

    forecast_status = "PASS"

    forecast_reasons.append(
        "threshold already crossed in the latest observation"
    )

elif not positive_growth:

    forecast_status = "FAIL"

    forecast_reasons.append(
        "estimated resistance growth is not positive"
    )

elif not valid_future_crossing:

    forecast_status = "FAIL"

    forecast_reasons.append(
        "the estimated threshold-crossing date is not after the notebook run date"
    )

elif (
    valid_bootstrap_fraction
    < BOOTSTRAP_CAUTION_FRACTION
):

    forecast_status = "FAIL"

    forecast_reasons.append(
        "fewer than 60% of bootstrap replicates produced future crossings"
    )

elif (
    not positive_growth_supported
    or valid_bootstrap_fraction
    < BOOTSTRAP_PASS_FRACTION
):

    forecast_status = "CAUTION"

    if not positive_growth_supported:

        forecast_reasons.append(
            "positive slope is not supported by its 95% confidence interval"
        )

    if (
        valid_bootstrap_fraction
        < BOOTSTRAP_PASS_FRACTION
    ):

        forecast_reasons.append(
            "60–79% of bootstrap replicates produced future crossings"
        )

else:

    forecast_status = "PASS"

    forecast_reasons.append(
        "positive growth and stable bootstrap crossing estimates"
    )

forecast_suitable = (
    forecast_status
    in ["PASS", "CAUTION"]
)

print("AMRTF forecast gate")
print("-------------------")
print(
    "Status:",
    forecast_status
)
print(
    "Reason:",
    "; ".join(forecast_reasons)
)
print()

# ============================================================
# 9. THRESHOLD FORECAST
# ============================================================
# Crossing quantities were calculated before the bootstrap so that the same
# values are used consistently by the gate, report, and plot.

# ============================================================
# 10. FINAL REPORT
# ============================================================

print("AMR FORECAST REPORT")
print("-------------------")
print("Pathogen:", pathogen)
print("Antibiotic:", antibiotic)
print("Country:", country)
print("Data source:", selected_source["source_name"])
print()

print("Notebook run date:", run_date.date())
print("Latest surveillance date:", latest_date.date())
print(
    "Latest resistance:",
    round(latest_R * 100, 2),
    "%"
)

print(
    "Estimated log O₀ (intercept):",
    round(log_O0, 5)
)

print(
    "Estimated g_R:",
    round(g_R, 5),
    "per year"
)

if (
    np.isfinite(slope_ci_lower)
    and np.isfinite(slope_ci_upper)
):
    print(
        "Slope 95% CI:",
        round(slope_ci_lower, 5),
        "to",
        round(slope_ci_upper, 5)
    )
else:
    print("Slope 95% CI: not available")

print(
    "Model R² (descriptive only):",
    round(r2, 3)
)

print(
    "Threshold:",
    round(theta * 100, 1),
    "%"
)

print()

print(
    "Reporting frequency:",
    reporting_frequency
)

print(
    "Observation span:",
    round(year_span, 2),
    "years"
)

print(
    "Observations:",
    n_observations
)

print(
    "Distinct observation years:",
    distinct_years
)

print(
    "Missing reporting periods:",
    missing_periods
)

print(
    "Missing fraction:",
    round(missing_fraction, 3)
)

print(
    "Reporting-delay rows removed:",
    reporting_delay_rows_removed
)

print()

print(
    "Forecast gate:",
    forecast_status
)

print(
    "Gate reason:",
    "; ".join(forecast_reasons)
)

if bootstrap_assessed:
    print(
        "Bootstrap success rate:",
        f"{valid_bootstrap_fraction * 100:.1f}%"
    )
else:
    print(
        "Bootstrap success rate: not assessed"
    )

print()

if threshold_already_crossed:

    print(
        "Threshold forecast status: "
        "threshold already crossed"
    )

    print(
        "Forecast interpretation: "
        "no future threshold-crossing forecast is required."
    )

    print(
        "Remaining clinical lifetime: 0 years"
    )

elif forecast_status == "FAIL":

    print(
        "Threshold forecast status: FAIL"
    )

    print(
        "Forecast interpretation: "
        "threshold forecast not recommended."
    )

    print(
        "No threshold-crossing forecast reported."
    )

else:

    print(
        "Threshold forecast status:",
        forecast_status
    )

    if forecast_status == "PASS":
        print(
            "Forecast interpretation: "
            "suitable for threshold forecasting."
        )

    elif forecast_status == "CAUTION":
        print(
            "Forecast interpretation: "
            "interpret the threshold forecast with caution."
        )

    print(
        "Predicted threshold crossing date:",
        crossing_date.date()
    )

    print(
        "Remaining clinical lifetime from run date:",
        round(remaining_from_run_date, 2),
        "years"
    )

    if len(bootstrap_remaining_years) > 0:

        lower_remaining = np.percentile(
            bootstrap_remaining_years,
            2.5
        )

        upper_remaining = np.percentile(
            bootstrap_remaining_years,
            97.5
        )

        lower_crossing_ordinal = np.percentile(
            [
                date.toordinal()
                for date
                in bootstrap_crossing_dates
            ],
            2.5
        )

        upper_crossing_ordinal = np.percentile(
            [
                date.toordinal()
                for date
                in bootstrap_crossing_dates
            ],
            97.5
        )

        lower_crossing_date = (
            pd.Timestamp.fromordinal(
                int(lower_crossing_ordinal)
            )
        )

        upper_crossing_date = (
            pd.Timestamp.fromordinal(
                int(upper_crossing_ordinal)
            )
        )

        print(
            "Bootstrap 95% remaining-lifetime interval:",
            round(lower_remaining, 2),
            "to",
            round(upper_remaining, 2),
            "years from run date"
        )

        print(
            "Bootstrap 95% crossing-date interval:",
            lower_crossing_date.date(),
            "to",
            upper_crossing_date.date()
        )

    else:
        print(
            "Bootstrap uncertainty interval: "
            "not available"
        )

print()

if selected_source["source_name"] == "UKHSA":

    print(
        "Note: The confidence region uses a residual bootstrap. "
        "UKHSA values are monthly rolling percentages, "
        "so uncertainty should be interpreted cautiously."
    )

elif selected_source["source_name"] == "JANIS":

    print(
        "Note: The confidence region uses a residual bootstrap. "
        "JANIS values are annual phenotypic resistance fractions "
        "calculated from resistant and tested isolate counts."
    )

# ============================================================
# 11. PUBLICATION-QUALITY FORECAST PLOT
# Extends forecast and bootstrap interval beyond threshold crossing
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ------------------------------------------------------------
# 1. Define extended plotting period
# ------------------------------------------------------------

observed_start = pd.to_datetime(data["date"].min())
observed_end = pd.to_datetime(data["date"].max())

if crossing_date is not None and forecast_status in ["PASS", "CAUTION"]:
    forecast_end = pd.to_datetime(crossing_date) + pd.DateOffset(months=6)
else:
    forecast_end = observed_end + pd.DateOffset(years=5)

# Monthly plotting grid, suitable for both UKHSA and JANIS figures
extended_dates = pd.date_range(
    start=observed_start,
    end=forecast_end,
    freq="MS"
)

# Time in continuous years from the same baseline used by the model
extended_t = (
    extended_dates - observed_start
).days.to_numpy() / 365.25

# ------------------------------------------------------------
# 2. Calculate fitted resistance trajectory
# ------------------------------------------------------------

# Change these two names only if your fitted coefficients use
# different variable names in Cell 3.
fitted_intercept = log_O0
fitted_slope = g_R

extended_log_odds = fitted_intercept + fitted_slope * extended_t
extended_R = 1.0 / (1.0 + np.exp(-extended_log_odds))

# ------------------------------------------------------------
# 3. Recalculate residual-bootstrap confidence interval
# ------------------------------------------------------------

# These should contain the accepted intercept and slope from
# each successful bootstrap replicate.
bootstrap_intercepts_array = np.asarray(bootstrap_intercepts, dtype=float)
bootstrap_slopes_array = np.asarray(bootstrap_slopes, dtype=float)

valid_bootstrap = (
    np.isfinite(bootstrap_intercepts_array)
    & np.isfinite(bootstrap_slopes_array)
)

bootstrap_intercepts_array = bootstrap_intercepts_array[valid_bootstrap]
bootstrap_slopes_array = bootstrap_slopes_array[valid_bootstrap]

if len(bootstrap_intercepts_array) > 0:
    bootstrap_log_odds = (
        bootstrap_intercepts_array[:, None]
        + bootstrap_slopes_array[:, None] * extended_t[None, :]
    )

    bootstrap_R = 1.0 / (1.0 + np.exp(-bootstrap_log_odds))

    extended_lower_band = np.percentile(
        bootstrap_R, 2.5, axis=0
    )

    extended_upper_band = np.percentile(
        bootstrap_R, 97.5, axis=0
    )
else:
    extended_lower_band = None
    extended_upper_band = None

# ------------------------------------------------------------
# 4. Separate fitted and extrapolated periods
# ------------------------------------------------------------

observed_period = extended_dates <= observed_end
forecast_period = extended_dates >= observed_end

# ------------------------------------------------------------
# 5. Create publication-quality figure
# ------------------------------------------------------------

figure_dir = Path("amr_alert_figures")
figure_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8.5, 5.5))

# Observed surveillance values
ax.scatter(
    data["date"],
    data["R"] * 100,
    s=34,
    label="Observed",
    zorder=5
)

# Fitted trajectory during the observed period
ax.plot(
    extended_dates[observed_period],
    extended_R[observed_period] * 100,
    linewidth=2.2,
    label="Fitted model",
    zorder=4
)

# Extrapolated forecast after the final observation
ax.plot(
    extended_dates[forecast_period],
    extended_R[forecast_period] * 100,
    linewidth=2.2,
    linestyle="--",
    label="Projected model",
    zorder=4
)

# Extended bootstrap interval
if (
    extended_lower_band is not None
    and extended_upper_band is not None
):
    ax.fill_between(
        extended_dates,
        extended_lower_band * 100,
        extended_upper_band * 100,
        alpha=0.15,
        linewidth=0,
        label="95% bootstrap interval",
        zorder=2
    )

# Clinical threshold
ax.axhline(
    theta * 100,
    linestyle="--",
    linewidth=1.8,
    label=f"{theta * 100:.0f}% threshold",
    zorder=1
)

# Threshold-crossing date
if crossing_date is not None and forecast_status in ["PASS", "CAUTION"]:
    crossing_date_plot = pd.to_datetime(crossing_date)

    ax.axvline(
        crossing_date_plot,
        linestyle=":",
        linewidth=1.8,
        label="Threshold crossing",
        zorder=1
    )

    ax.annotate(
        crossing_date_plot.strftime("%b %Y"),
        xy=(crossing_date_plot, theta * 100),
        xytext=(-7, -18),
        textcoords="offset points",
        ha="right",
        va="top",
        fontsize=9
    )

# ------------------------------------------------------------
# 6. Formatting
# ------------------------------------------------------------

ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Resistance (%)", fontsize=11)

ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.tick_params(axis="both", labelsize=10)

ax.grid(
    True,
    linewidth=0.6,
    alpha=0.30
)

ax.set_ylim(
    bottom=0,
    top=max(
        theta * 100 * 1.12,
        np.nanmax(extended_upper_band * 100)
        if extended_upper_band is not None
        else np.nanmax(extended_R * 100) * 1.08
    )
)

# Legend below the plot so it does not cover observations
ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.13),
    ncol=3,
    frameon=False,
    fontsize=9
)

fig.tight_layout()

# ------------------------------------------------------------
# 7. Save publication files
# ------------------------------------------------------------

base_name = "Figure_1_AMR_ALERT_Ecoli_piperacillin_tazobactam_extended"

png_path = figure_dir / f"{base_name}.png"
pdf_path = figure_dir / f"{base_name}.pdf"
svg_path = figure_dir / f"{base_name}.svg"

fig.savefig(png_path, dpi=600, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(svg_path, bbox_inches="tight")

plt.show()

print("Saved publication-quality figure:")
print(f"PNG: {png_path}")
print(f"PDF: {pdf_path}")
print(f"SVG: {svg_path}")

In [ ]:
#@title AMR Forecast — Batch Suitability Summary Cell-4

import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# 1. BATCH OUTPUT COLUMNS
# ============================================================

BATCH_COLUMNS = [
    "pathogen",
    "antibiotic",
    "country",
    "latest_date",
    "latest_R_percent",
    "g_R_slope",
    "slope_CI_lower",
    "slope_CI_upper",
    "log_O0_intercept",
    "R2",
    "bootstrap_success_percent",
    "threshold_status",
    "predicted_crossing_date",
    "remaining_lifetime_years",
    "forecast_status",
    "gate_reason",
]


# ============================================================
# 2. BATCH GATE CONSTANTS
# ============================================================

MIN_OBSERVATIONS = 8
MIN_DISTINCT_YEARS = 3
MIN_RESIDUAL_DF = 4
MIN_DISTINCT_R_VALUES = 3

BOOTSTRAP_PASS_FRACTION = 0.80
BOOTSTRAP_CAUTION_FRACTION = 0.60

N_BOOTSTRAP_BATCH = 500
BATCH_RANDOM_SEED = 123


# ============================================================
# 3. GENERAL HELPERS
# ============================================================

def get_batch_pathogens(country):
    matches = source_registry[
        source_registry["country"].astype(str).str.lower().eq(
            str(country).lower()
        )
        &
        source_registry["status"].astype(str).str.lower().eq("supported")
    ]

    return sorted(
        matches["pathogen"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )


def find_column(df, candidates):
    lookup = {
        str(column).strip().lower(): column
        for column in df.columns
    }

    for candidate in candidates:
        key = str(candidate).strip().lower()

        if key in lookup:
            return lookup[key]

    return None


def extract_latest_janis_raw_value(raw_df, registry_row):
    """
    Recover the latest JANIS date and R when the JANIS standardizer
    rejects a series containing R = 0 or R = 1.
    """
    if not isinstance(raw_df, pd.DataFrame):
        return None, None

    df = raw_df.copy()

    filter_candidates = {
        "pathogen": ["pathogen", "organism", "bacterium", "species"],
        "antibiotic": ["antibiotic", "antimicrobial", "drug"],
        "country": ["country", "geography"],
        "specimen": ["specimen", "sample", "specimen_type"],
    }

    for registry_name, candidates in filter_candidates.items():
        column = find_column(df, candidates)
        value = registry_row.get(registry_name, None)

        if column is not None and value is not None and pd.notna(value):
            df = df[
                df[column]
                .astype(str)
                .str.strip()
                .str.lower()
                .eq(str(value).strip().lower())
            ]

    if df.empty:
        return None, None

    date_column = find_column(
        df,
        ["date", "observation_date", "report_date"],
    )

    year_column = find_column(
        df,
        ["year", "calendar_year"],
    )

    if date_column is not None:
        df["_fallback_date"] = pd.to_datetime(
            df[date_column],
            errors="coerce",
        )

    elif year_column is not None:
        years = pd.to_numeric(
            df[year_column],
            errors="coerce",
        )

        df["_fallback_date"] = pd.to_datetime(
            years.astype("Int64").astype(str) + "-01-01",
            errors="coerce",
        )

    else:
        return None, None

    percent_column = find_column(
        df,
        [
            "resistance_percent",
            "resistant_percent",
            "resistance_percentage",
            "percent_resistant",
            "r_percent",
            "r_pct",
            "resistance_rate_percent",
        ],
    )

    fraction_column = find_column(
        df,
        [
            "R",
            "resistance_fraction",
            "resistant_fraction",
            "proportion_resistant",
            "resistance_rate",
        ],
    )

    resistant_column = find_column(
        df,
        [
            "resistant",
            "resistant_count",
            "n_resistant",
            "number_resistant",
        ],
    )

    tested_column = find_column(
        df,
        [
            "tested",
            "tested_count",
            "n_tested",
            "total",
            "total_tested",
            "number_tested",
        ],
    )

    if percent_column is not None:
        df["_fallback_R"] = (
            pd.to_numeric(
                df[percent_column],
                errors="coerce",
            )
            / 100.0
        )

    elif fraction_column is not None:
        values = pd.to_numeric(
            df[fraction_column],
            errors="coerce",
        )

        finite_values = values.dropna()

        if not finite_values.empty and finite_values.max() > 1:
            values = values / 100.0

        df["_fallback_R"] = values

    elif resistant_column is not None and tested_column is not None:
        resistant = pd.to_numeric(
            df[resistant_column],
            errors="coerce",
        )

        tested = pd.to_numeric(
            df[tested_column],
            errors="coerce",
        )

        df["_fallback_R"] = np.where(
            tested > 0,
            resistant / tested,
            np.nan,
        )

    else:
        return None, None

    df = (
        df.dropna(
            subset=[
                "_fallback_date",
                "_fallback_R",
            ]
        )
        .sort_values("_fallback_date")
        .reset_index(drop=True)
    )

    if df.empty:
        return None, None

    latest = df.iloc[-1]

    return (
        latest["_fallback_date"].date(),
        float(latest["_fallback_R"]),
    )


def make_batch_row(
    source_row,
    latest_date=None,
    latest_R_percent=None,
    slope=None,
    slope_ci_lower=None,
    slope_ci_upper=None,
    intercept=None,
    r2=None,
    bootstrap_success_percent=None,
    threshold_status=None,
    predicted_crossing_date=None,
    remaining_lifetime_years=None,
    forecast_status="ERROR",
    gate_reason=None,
):
    return {
        "pathogen": source_row["pathogen"],
        "antibiotic": source_row["antibiotic"],
        "country": source_row["country"],
        "latest_date": latest_date,
        "latest_R_percent": latest_R_percent,
        "g_R_slope": slope,
        "slope_CI_lower": slope_ci_lower,
        "slope_CI_upper": slope_ci_upper,
        "log_O0_intercept": intercept,
        "R2": r2,
        "bootstrap_success_percent": bootstrap_success_percent,
        "threshold_status": threshold_status,
        "predicted_crossing_date": predicted_crossing_date,
        "remaining_lifetime_years": remaining_lifetime_years,
        "forecast_status": forecast_status,
        "gate_reason": gate_reason,
    }


# ============================================================
# 4. UNIFIED AMRTF BATCH GATE
# ============================================================

def assess_amrtf_batch_series(data, threshold, run_date, random_seed):
    """
    Apply the same model-based AMRTF gate used in Cell 3.

    Returns a dictionary containing model estimates, bootstrap stability,
    threshold-crossing results, and PASS/CAUTION/FAIL classification.
    """
    data = data.copy()

    data["date"] = pd.to_datetime(
        data["date"],
        errors="coerce",
    )

    data["R"] = pd.to_numeric(
        data["R"],
        errors="coerce",
    )

    data = (
        data.dropna(subset=["date", "R"])
        .sort_values("date")
        .drop_duplicates(
            subset=["date"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    if data.empty:
        raise ValueError("No valid standardized observations.")

    if ((data["R"] <= 0) | (data["R"] >= 1)).any():
        raise ValueError(
            "Resistance fractions must be strictly between 0 and 1 "
            "for the log-odds model."
        )

    latest_date = data["date"].max()

    latest_R = float(
        data.loc[
            data["date"].eq(latest_date),
            "R",
        ].iloc[-1]
    )

    n_observations = len(data)
    distinct_years = data["date"].dt.year.nunique()
    distinct_R_values = data["R"].nunique()

    data["log_O"] = np.log(
        data["R"] / (1 - data["R"])
    )

    start_date = data["date"].min()

    data["t_years"] = (
        data["date"] - start_date
    ).dt.days / 365.25

    if data["t_years"].nunique() < 2:
        raise ValueError(
            "At least two distinct observation times are required."
        )

    fit = linregress(
        data["t_years"],
        data["log_O"],
    )

    slope = float(fit.slope)
    intercept = float(fit.intercept)
    slope_se = float(fit.stderr)

    predictions = (
        intercept
        + slope * data["t_years"]
    )

    r2 = float(
        r2_score(
            data["log_O"],
            predictions,
        )
    )

    residual_df = n_observations - 2

    if np.isfinite(slope_se) and residual_df > 0:
        slope_ci_lower = slope - 1.96 * slope_se
        slope_ci_upper = slope + 1.96 * slope_se
    else:
        slope_ci_lower = np.nan
        slope_ci_upper = np.nan

    basic_estimability = (
        n_observations >= MIN_OBSERVATIONS
        and distinct_years >= MIN_DISTINCT_YEARS
        and residual_df >= MIN_RESIDUAL_DF
        and distinct_R_values >= MIN_DISTINCT_R_VALUES
        and data["t_years"].nunique() >= 3
        and np.isfinite(slope)
        and np.isfinite(intercept)
        and np.isfinite(slope_se)
    )

    positive_growth = (
        np.isfinite(slope)
        and slope > 0
    )

    positive_growth_supported = (
        positive_growth
        and np.isfinite(slope_ci_lower)
        and slope_ci_lower > 0
    )

    threshold_already_crossed = (
        latest_R >= threshold
    )

    target_log_odds = np.log(
        threshold / (1 - threshold)
    )

    crossing_date = None
    remaining_lifetime_years = None
    valid_future_crossing = False

    if (
        basic_estimability
        and positive_growth
        and not threshold_already_crossed
    ):
        crossing_time_years = (
            target_log_odds - intercept
        ) / slope

        if np.isfinite(crossing_time_years):
            crossing_date = (
                start_date
                + pd.to_timedelta(
                    crossing_time_years * 365.25,
                    unit="D",
                )
            )

            remaining_lifetime_years = (
                crossing_date - run_date
            ).days / 365.25

            valid_future_crossing = (
                np.isfinite(remaining_lifetime_years)
                and remaining_lifetime_years >= 0
            )

    bootstrap_assessed = (
        basic_estimability
        and positive_growth
        and not threshold_already_crossed
        and valid_future_crossing
    )

    valid_bootstrap_crossings = 0

    if bootstrap_assessed:
        rng = np.random.default_rng(random_seed)

        residuals = (
            data["log_O"] - predictions
        ).to_numpy()

        fitted_values = predictions.to_numpy()
        t_values = data["t_years"].to_numpy()

        for _ in range(N_BOOTSTRAP_BATCH):
            sampled_residuals = rng.choice(
                residuals,
                size=len(residuals),
                replace=True,
            )

            boot_y = (
                fitted_values
                + sampled_residuals
            )

            boot_fit = linregress(
                t_values,
                boot_y,
            )

            boot_slope = boot_fit.slope
            boot_intercept = boot_fit.intercept

            if not (
                np.isfinite(boot_slope)
                and np.isfinite(boot_intercept)
                and boot_slope > 0
            ):
                continue

            boot_crossing_time = (
                target_log_odds - boot_intercept
            ) / boot_slope

            if not np.isfinite(boot_crossing_time):
                continue

            boot_crossing_date = (
                start_date
                + pd.to_timedelta(
                    boot_crossing_time * 365.25,
                    unit="D",
                )
            )

            boot_remaining = (
                boot_crossing_date - run_date
            ).days / 365.25

            if (
                np.isfinite(boot_remaining)
                and boot_remaining >= 0
            ):
                valid_bootstrap_crossings += 1

        valid_bootstrap_fraction = (
            valid_bootstrap_crossings
            / N_BOOTSTRAP_BATCH
        )

    else:
        valid_bootstrap_fraction = None

    reasons = []

    if not basic_estimability:
        forecast_status = "FAIL"
        reasons.append(
            "insufficient information for stable model estimation"
        )

    elif threshold_already_crossed:
        forecast_status = "PASS"
        reasons.append(
            "threshold already crossed in the latest observation"
        )

    elif not positive_growth:
        forecast_status = "FAIL"
        reasons.append(
            "estimated resistance growth is not positive"
        )

    elif not valid_future_crossing:
        forecast_status = "FAIL"
        reasons.append(
            "estimated crossing date is not after the notebook run date"
        )

    elif (
        valid_bootstrap_fraction
        < BOOTSTRAP_CAUTION_FRACTION
    ):
        forecast_status = "FAIL"
        reasons.append(
            "fewer than 60% of bootstrap replicates produced future crossings"
        )

    elif (
        not positive_growth_supported
        or valid_bootstrap_fraction
        < BOOTSTRAP_PASS_FRACTION
    ):
        forecast_status = "CAUTION"

        if not positive_growth_supported:
            reasons.append(
                "positive slope is not supported by its 95% confidence interval"
            )

        if (
            valid_bootstrap_fraction
            < BOOTSTRAP_PASS_FRACTION
        ):
            reasons.append(
                "60–79% of bootstrap replicates produced future crossings"
            )

    else:
        forecast_status = "PASS"
        reasons.append(
            "positive growth and stable bootstrap crossing estimates"
        )

    if threshold_already_crossed:
        threshold_status = "ALREADY CROSSED"
        predicted_crossing_date = None
        remaining_lifetime_years = 0.0

    elif forecast_status == "FAIL":
        threshold_status = "NOT REPORTED"
        predicted_crossing_date = None
        remaining_lifetime_years = None

    else:
        threshold_status = "FUTURE CROSSING"
        predicted_crossing_date = crossing_date.date()

    return {
        "latest_date": latest_date.date(),
        "latest_R_percent": round(latest_R * 100, 2),
        "slope": round(slope, 5),
        "slope_ci_lower": (
            round(float(slope_ci_lower), 5)
            if np.isfinite(slope_ci_lower)
            else None
        ),
        "slope_ci_upper": (
            round(float(slope_ci_upper), 5)
            if np.isfinite(slope_ci_upper)
            else None
        ),
        "intercept": round(intercept, 5),
        "r2": round(r2, 3),
        "bootstrap_success_percent": (
            round(valid_bootstrap_fraction * 100, 1)
            if valid_bootstrap_fraction is not None
            else None
        ),
        "threshold_status": threshold_status,
        "predicted_crossing_date": predicted_crossing_date,
        "remaining_lifetime_years": (
            round(float(remaining_lifetime_years), 2)
            if remaining_lifetime_years is not None
            else None
        ),
        "forecast_status": forecast_status,
        "gate_reason": "; ".join(reasons),
    }


# ============================================================
# 5. RUN BATCH SUMMARY
# ============================================================

def run_batch_summary(country, pathogen, threshold):
    batch_results = []

    batch_matches = source_registry[
        source_registry["country"].astype(str).str.lower().eq(
            str(country).lower()
        )
        &
        source_registry["pathogen"].astype(str).str.lower().eq(
            str(pathogen).lower()
        )
        &
        source_registry["status"].astype(str).str.lower().eq(
            "supported"
        )
    ].copy()

    if batch_matches.empty:
        return pd.DataFrame(columns=BATCH_COLUMNS)

    run_date = (
        pd.Timestamp.now(tz="Europe/London")
        .normalize()
        .tz_localize(None)
    )

    for row_number, (_, source_row) in enumerate(
        batch_matches.iterrows()
    ):
        fallback_date = None
        fallback_R = None

        try:
            raw_data = source_row["loader"](
                source_row["url"]
            )

            if country.lower() == "japan":
                fallback_date, fallback_R = (
                    extract_latest_janis_raw_value(
                        raw_data,
                        source_row,
                    )
                )

            data = source_row["standardizer"](
                raw_data,
                source_row,
            ).copy()

            missing_columns = {
                "date",
                "R",
            } - set(data.columns)

            if missing_columns:
                raise ValueError(
                    "Missing standardized columns: "
                    f"{sorted(missing_columns)}"
                )

            result = assess_amrtf_batch_series(
                data=data,
                threshold=threshold,
                run_date=run_date,
                random_seed=BATCH_RANDOM_SEED + row_number,
            )

            batch_results.append(
                make_batch_row(
                    source_row,
                    latest_date=result["latest_date"],
                    latest_R_percent=result["latest_R_percent"],
                    slope=result["slope"],
                    slope_ci_lower=result["slope_ci_lower"],
                    slope_ci_upper=result["slope_ci_upper"],
                    intercept=result["intercept"],
                    r2=result["r2"],
                    bootstrap_success_percent=(
                        result["bootstrap_success_percent"]
                    ),
                    threshold_status=result["threshold_status"],
                    predicted_crossing_date=(
                        result["predicted_crossing_date"]
                    ),
                    remaining_lifetime_years=(
                        result["remaining_lifetime_years"]
                    ),
                    forecast_status=result["forecast_status"],
                    gate_reason=result["gate_reason"],
                )
            )

        except Exception as error:
            error_message = str(error)

            if (
                "Resistance fractions must be strictly between 0 and 1"
                in error_message
            ):
                fallback_percent = None

                if fallback_R is not None:
                    fallback_percent = max(
                        round(
                            float(fallback_R) * 100,
                            2,
                        ),
                        0.01,
                    )

                batch_results.append(
                    make_batch_row(
                        source_row,
                        latest_date=fallback_date,
                        latest_R_percent=fallback_percent,
                        threshold_status="NOT REPORTED",
                        forecast_status="FAIL",
                        gate_reason=(
                            "series contains resistance values equal to 0 or 1"
                        ),
                    )
                )

            else:
                print(
                    "Batch error | "
                    f"{source_row['pathogen']} | "
                    f"{source_row['antibiotic']} | "
                    f"{source_row['country']} | "
                    f"{error}"
                )

                batch_results.append(
                    make_batch_row(
                        source_row,
                        forecast_status="ERROR",
                        gate_reason=error_message,
                    )
                )

    summary = (
        pd.DataFrame(batch_results)
        .reindex(columns=BATCH_COLUMNS)
    )

    if summary.empty:
        return summary

    status_order = pd.CategoricalDtype(
        categories=[
            "PASS",
            "CAUTION",
            "FAIL",
            "ERROR",
        ],
        ordered=True,
    )

    summary["forecast_status"] = (
        summary["forecast_status"]
        .astype(status_order)
    )

    summary = (
        summary.sort_values(
            [
                "forecast_status",
                "antibiotic",
            ]
        )
        .reset_index(drop=True)
    )

    summary["forecast_status"] = (
        summary["forecast_status"]
        .astype(str)
    )

    return summary


# ============================================================
# 6. DYNAMIC CELL 4 CONTROLS
# ============================================================

available_countries = [
    country_name
    for country_name in ["England", "Japan"]
    if get_batch_pathogens(country_name)
]

batch_country_widget = widgets.Dropdown(
    options=available_countries,
    value=available_countries[0],
    description="Country:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="320px"),
)

initial_pathogens = get_batch_pathogens(
    batch_country_widget.value
)

batch_pathogen_widget = widgets.Dropdown(
    options=initial_pathogens,
    value=initial_pathogens[0],
    description="Pathogen:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="480px"),
)

batch_threshold_widget = widgets.Dropdown(
    options=[10, 20, 30, 40, 50],
    value=int(round(theta * 100)),
    description="Threshold:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="320px"),
)

batch_run_button = widgets.Button(
    description="Run batch summary",
    button_style="primary",
    icon="play",
)

batch_output = widgets.Output()


def update_pathogen_dropdown(change):
    pathogens = get_batch_pathogens(
        change["new"]
    )

    batch_pathogen_widget.options = pathogens
    batch_pathogen_widget.value = (
        pathogens[0]
        if pathogens
        else None
    )

    batch_run_button.disabled = not bool(
        pathogens
    )

    with batch_output:
        clear_output(wait=True)


def run_batch_button_clicked(_):
    with batch_output:
        clear_output(wait=True)

        selected_country = batch_country_widget.value
        selected_pathogen = batch_pathogen_widget.value
        selected_threshold = (
            batch_threshold_widget.value / 100
        )

        if selected_pathogen is None:
            print(
                "No supported pathogen is available "
                "for the selected country."
            )
            return

        run_date = (
            pd.Timestamp.now(tz="Europe/London")
            .normalize()
            .tz_localize(None)
        )

        print(
            f"Country: {selected_country}"
        )

        print(
            f"Pathogen: {selected_pathogen}"
        )

        print(
            f"Threshold: {selected_threshold * 100:.1f}%"
        )

        print(
            f"Notebook run date: {run_date.date()}"
        )

        summary = run_batch_summary(
            country=selected_country,
            pathogen=selected_pathogen,
            threshold=selected_threshold,
        )

        display(summary)


batch_country_widget.observe(
    update_pathogen_dropdown,
    names="value",
)

batch_run_button.on_click(
    run_batch_button_clicked
)


# ============================================================
# 7. DISPLAY CELL 4
# ============================================================

display(
    widgets.VBox(
        [
            widgets.HTML(
                "<h3>AMR Forecast — "
                "Batch Suitability Summary</h3>"
            ),
            batch_country_widget,
            batch_pathogen_widget,
            batch_threshold_widget,
            batch_run_button,
            batch_output,
        ]
    )
)